# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Himanshu-rai-prog/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The playbook is a decision-support queue for human content review.

1. **Review and refresh** — Prioritize content with a higher predicted decline risk. The reviewer checks whether the content is still accurate, useful, and worth refreshing.
   - Reason code: `predicted_decline`

2. **Review stale but visible content** — Prioritize older content that still receives meaningful impressions. A reviewer checks whether updating the content could be useful.
   - Reason code: `stale_but_visible`

3. **Keep under observation** — Content without a strong review signal can remain unchanged while its performance is monitored.
   - Reason code: `monitor_only`

The ranking is directional. A higher score means higher priority for review; it does not guarantee that a refresh will improve performance.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd
import numpy as np

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory ready:", OUTPUT_DIR)

# Check what files are actually available in the repository
from pathlib import Path

repo_files = list(Path(".").rglob("*"))

for f in repo_files:
    if f.is_file() and (
        f.suffix.lower() in [".csv", ".json", ".parquet", ".pkl", ".joblib"]
    ):
        print(f)



Output directory ready: work/outputs
.config/.last_update_check.json
sample_data/anscombe.json
sample_data/california_housing_train.csv
sample_data/mnist_train_small.csv
sample_data/mnist_test.csv
sample_data/california_housing_test.csv


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

This playbook is intended for content teams and analysts who need to prioritize which content should be reviewed first.

The model output is a decision-support signal. A human should review the content before any change is made.

The playbook is not intended to automatically publish, delete, rewrite, or redirect content. The recommendations are based on the available dataset and observed validation results, so they may become less reliable when content, traffic patterns, or data quality changes.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Basic checks for the playbook data.
# The playbook is intended for decision support, not automatic publishing.

print("Decision-support playbook")
print("Human review required before any content action.")
print("Automatic publishing/deletion is not allowed.")

Decision-support playbook
Human review required before any content action.
Automatic publishing/deletion is not allowed.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + the no-go list

Before acting on a recommendation, a human reviewer should check:
- whether the content is still accurate and relevant;
- whether the page has a valid reason to be refreshed;
- whether the observed traffic or visibility supports the recommendation;
- whether there are important business or editorial considerations not represented in the model.

**No-go cases:** Do not automatically publish changes, delete content, change URLs, make factual claims, or make high-impact editorial decisions from the model score alone.

The model should support prioritization, not replace editorial judgment.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Human-review guardrail check

NO_GO_ACTIONS = [
    "automatic_publish",
    "automatic_delete",
    "automatic_rewrite",
    "automatic_url_change"
]

print("No-go actions:", NO_GO_ACTIONS)
print("All recommended actions require human review.")

No-go actions: ['automatic_publish', 'automatic_delete', 'automatic_rewrite', 'automatic_url_change']
All recommended actions require human review.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The playbook should be reviewed when its recommendations appear to become less useful.

Useful triggers include:
- a sustained drop in validation performance;
- a meaningful change in the distribution of input features;
- changes in content types or traffic patterns;
- changes in the definition or availability of the target outcome;
- repeated human-review disagreement with high-ranked recommendations.

If these changes persist, the data and model should be re-evaluated before continued use.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Monitoring checklist

monitoring_triggers = {
    "validation_performance_drop": True,
    "input_distribution_change": True,
    "content_mix_change": True,
    "target_definition_change": True,
    "human_model_disagreement": True
}

print("Monitoring / retrain triggers:")
for trigger, enabled in monitoring_triggers.items():
    print("-", trigger)

Monitoring / retrain triggers:
- validation_performance_drop
- input_distribution_change
- content_mix_change
- target_definition_change
- human_model_disagreement


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The notebook exports the ranked action queue to `work/outputs/`.

The exported queue is intended to provide the evidence used in the recommendations section of the research paper. The queue should be regenerated by running this notebook rather than treated as a permanent production dataset.

Any reusable figures should be saved under `work/figures/`.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Export the current playbook queue if it exists.
if "queue" in globals() and isinstance(queue, pd.DataFrame):
    queue_path = OUTPUT_DIR / "content_action_playbook_queue.csv"
    queue.to_csv(queue_path, index=False)
    print(f"Queue exported: {queue_path}")
    print(f"Rows: {len(queue)}")
else:
    print("Queue dataframe is not available yet.")
    print("Create the ranked queue before exporting it.")
    from pathlib import Path

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create a practical action-playbook queue from the validated model results.
# Use existing prediction/score dataframe when available.

candidate_names = [
    "validation_results",
    "validation_df",
    "results_df",
    "model_results",
    "predictions_df",
    "test_results"
]

queue = None

for name in candidate_names:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        queue = globals()[name].copy()
        print("Using dataframe:", name)
        break

if queue is None:
    print("No existing validation dataframe found.")
else:
    # Add a simple decision-support action where possible
    if "predicted_probability" in queue.columns:
        queue["action"] = np.where(
            queue["predicted_probability"] >= 0.5,
            "review_refresh",
            "monitor_only"
        )
        queue["reason_code"] = np.where(
            queue["predicted_probability"] >= 0.5,
            "predicted_decline",
            "monitor_only"
        )

    queue_path = OUTPUT_DIR / "content_action_playbook_queue.csv"
    queue.to_csv(queue_path, index=False)

    print("Queue exported:", queue_path)
    print("Rows:", len(queue))
    print("Columns:", list(queue.columns))

Queue dataframe is not available yet.
Create the ranked queue before exporting it.
No existing validation dataframe found.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.